# imports

In [1]:
import os
os.chdir('/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl')

In [67]:
from importlib import reload
import interpretable_ssl.evaluation.metric_helpers.embedding_metrics
import interpretable_ssl.evaluation.metric_helpers.embedding_tables
reload(interpretable_ssl.evaluation.metric_helpers.embedding_metrics)
reload(interpretable_ssl.evaluation.metric_helpers.embedding_tables)

<module 'interpretable_ssl.evaluation.metric_helpers.embedding_tables' from '/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl/interpretable_ssl/evaluation/metric_helpers/embedding_tables.py'>

In [68]:
from interpretable_ssl.evaluation.metric_helpers.embedding_metrics import *
from interpretable_ssl.evaluation.metric_helpers.embedding_tables import *
from interpretable_ssl.datasets.dataset_configs import *

In [51]:
def save_baseline_metrics(ds_id, pt_epochs=50, ft_epochs=1):
    ds_conf = DATASETS[ds_id]
    adata = sc.read_h5ad(ds_conf["path"])
    if adata.X.max() > 50:
        print('adata x is not normalized')
    bk, lk = ds_conf["batch_key"], ds_conf["label_key"]
    add_pca_harmoney(adata, ds_conf["batch_key"], "X_pca")
    add_scvi_emb(adata, ds_conf["test_studies"], bk, pt_epochs, ft_epochs)
    scg, scb = get_metrics(adata, ['X_pca', 'X_pca_harmoney', 'X_scvi'], bk, lk)
    home = '/ictstr01/home/icb/fatemehs.hashemig/'
    os.makedirs(home + f'/models/{ds_id}/baselines/', exist_ok=True)
    scg.to_csv(home + f'/models/{ds_id}/baselines/scgraph.csv')
    scb.to_csv(home + f'/models/{ds_id}/baselines/scib.csv')
    mc_ad = load_seacell(ds_id)
    add_pca_harmoney(mc_ad, bk, 'seacell_pca')
    scg, scb = get_metacell_metrics(mc_ad, ['seacell_pca', 'seacell_pca_harmoney'], bk, lk)
    os.makedirs(home + f'/models/{ds_id}/seacell/', exist_ok=True)
    scg.to_csv(home + f'/models/{ds_id}/seacell/scgraph.csv')
    scb.to_csv(home + f'/models/{ds_id}/seacell/scib.csv')

# pancreas

In [69]:
df, dfs = load_tb('pancreas')

/home/icb/fatemehs.hashemig/models/pancreas/seacell_sc.h5ad/scib.csv
/home/icb/fatemehs.hashemig/models/pancreas/seacell_agg.h5ad/scib.csv
/home/icb/fatemehs.hashemig/models/pancreas/scpoli_ds_panc_finetune_20_cvae_pre_100/scib.csv
/home/icb/fatemehs.hashemig/models/pancreas/seacell_sc.h5ad/scgraph.csv
/home/icb/fatemehs.hashemig/models/pancreas/seacell_agg.h5ad/scgraph.csv
/home/icb/fatemehs.hashemig/models/pancreas/scpoli_ds_panc_finetune_20_cvae_pre_100/scgraph.csv


In [76]:
def is_mc(idx):
    return ('mc' in idx) or ('seacell' in idx)
mask = np.array([is_mc(idx) for idx in df.index])
df[mask]

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,Silhouette batch,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
swav_ds_panc_NP_220_bs_256_mc_pca,0.528825,0.660357,0.519134,0.611786,0.964370,0.804386,0.245743,0.815068,0.980038,0.000000e+00,0.569047,0.656894,0.621756,0.984694,0.989955,0.971395
swav_ds_panc_NP_220_bs_256_knn_diff_mc_pca,0.500000,0.654802,0.473306,0.612784,0.964310,0.758983,0.308999,0.960140,0.921589,0.000000e+00,0.589942,0.641040,0.620601,0.877551,0.976667,0.903867
seacell_pca,0.500000,0.553293,0.273967,0.589719,0.936285,0.692187,0.162420,0.640394,0.827775,3.260648e-07,0.464555,0.570653,0.528214,0.910714,0.991154,0.961740
seacell_pca_harmoney,0.500000,0.698510,0.474634,0.639701,0.967154,0.829590,0.343208,0.757881,0.820925,4.966550e-01,0.649652,0.656000,0.653461,0.880952,0.988750,0.961405
swav_ds_panc_NP_220_cvae_pre_100_bs_256_mc_pca,0.556721,0.715544,0.464228,0.672920,0.993561,0.795286,0.142232,0.980684,0.981720,0.000000e+00,0.579985,0.680595,0.640351,0.903061,0.979793,0.874967
swav_ds_panc_NP_220_finetune_20_cvae_pre_100_bs_256_mc_pca,0.500000,0.813209,0.665789,0.711648,0.998059,0.780039,0.185321,0.934862,0.907967,0.000000e+00,0.561638,0.737741,0.667300,0.954082,0.986896,0.947962
swav_ds_panc_NP_220_cvae_pre_100_bs_256_knn_diff_mc_pca,0.404299,0.770174,0.630192,0.671274,0.994051,0.749374,0.216408,0.945122,0.929058,0.000000e+00,0.567992,0.693998,0.643596,0.933673,0.968435,0.870570


In [78]:
show_tb(df[mask])

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
swav_ds_panc_NP_220_bs_256_mc_pca,0.569,0.657,0.622,0.985,0.990,0.971
swav_ds_panc_NP_220_bs_256_knn_diff_mc_pca,0.590,0.641,0.621,0.878,0.977,0.904
seacell_pca,0.465,0.571,0.528,0.911,0.991,0.962
seacell_pca_harmoney,0.650,0.656,0.653,0.881,0.989,0.961
swav_ds_panc_NP_220_cvae_pre_100_bs_256_mc_pca,0.580,0.681,0.640,0.903,0.980,0.875
swav_ds_panc_NP_220_finetune_20_cvae_pre_100_bs_256_mc_pca,0.562,0.738,0.667,0.954,0.987,0.948
swav_ds_panc_NP_220_cvae_pre_100_bs_256_knn_diff_mc_pca,0.568,0.694,0.644,0.934,0.968,0.871


In [79]:
show_tb(df[~mask])

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
scpoli_ds_panc_cvae_pre_100,0.365,0.658,0.541,0.633,0.701,0.543
swav_ds_panc_NP_220_finetune_20_cvae_pre_100_bs_256_knn_diff,0.604,0.703,0.663,0.381,0.658,0.319
swav_ds_panc_NP_220_bs_256,0.562,0.715,0.654,0.636,0.784,0.614
swav_ds_panc_NP_220_bs_256_knn_diff,0.586,0.712,0.662,0.588,0.767,0.557
X_pca,0.342,0.672,0.540,0.915,0.962,0.921
X_pca_harmoney,0.681,0.776,0.738,0.916,0.971,0.932
X_scvi,0.570,0.749,0.678,0.540,0.761,0.523
scpoli_ds_panc,0.372,0.651,0.540,0.554,0.616,0.429
swav_ds_panc_NP_220_cvae_pre_100_bs_256,0.617,0.731,0.685,0.464,0.689,0.375
swav_ds_panc_NP_220_finetune_20_cvae_pre_100_bs_256,0.591,0.706,0.660,0.456,0.698,0.386


In [74]:
show_tb(df[~mask])

TypeError: bad operand type for unary ~: 'list'

# immune

In [52]:
save_baseline_metrics('pbmc-immune')

2025-09-15 08:31:20,305 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO:harmonypy:Computing initial centroids with sklearn.KMeans...
2025-09-15 08:31:35,267 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO:harmonypy:sklearn.KMeans initialization complete.
2025-09-15 08:31:35,497 - harmonypy - INFO - Iteration 1 of 10
INFO:harmonypy:Iteration 1 of 10
2025-09-15 08:31:50,389 - harmonypy - INFO - Iteration 2 of 10
INFO:harmonypy:Iteration 2 of 10
2025-09-15 08:32:08,539 - harmonypy - INFO - Iteration 3 of 10
INFO:harmonypy:Iteration 3 of 10
2025-09-15 08:32:27,321 - harmonypy - INFO - Iteration 4 of 10
INFO:harmonypy:Iteration 4 of 10
2025-09-15 08:32:47,373 - harmonypy - INFO - Converged after 4 iterations
INFO:harmonypy:Converged after 4 iterations
Multiprocessing is handled by SLURM.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_R

Epoch 50/50: 100%|██████████| 50/50 [02:12<00:00,  2.72s/it, loss=647, v_num=1]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [02:12<00:00,  2.66s/it, loss=647, v_num=1]


Multiprocessing is handled by SLURM.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


...

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    



Metrics:   0%|          | 0/10 [00:00<?, ?it/s]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:14<00:06,  1.56s/it, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    



Embeddings: 100%|██████████| 3/3 [05:54<00:00, 118.30s/it]tch correction: pcr_comparison]

                                                                                         

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/home/icb/fatemehs.hashemig//models/pbmc-immune/seacell_agg.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [56]:
dfs = load_tb('pbmc-immune')
show_tb(dfs)

/home/icb/fatemehs.hashemig/models/pbmc-immune/scpoli_finetune_20_cvae_pre_100/scib.csv
/home/icb/fatemehs.hashemig/models/pbmc-immune/scpoli_cvae_pre_100/scib.csv
/home/icb/fatemehs.hashemig/models/pbmc-immune/scpoli_finetune_20_cvae_pre_100/scgraph.csv
/home/icb/fatemehs.hashemig/models/pbmc-immune/scpoli_cvae_pre_100/scgraph.csv


,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
X_pca,0.317,0.621,0.500,0.665,0.499,0.357
X_pca_harmoney,0.560,0.673,0.628,0.490,0.653,0.443
X_scvi,0.558,0.692,0.638,0.577,0.638,0.458
scpoli,0.530,0.554,0.545,0.567,0.651,0.540
swav_knn_diff_mc_pca,0.591,0.491,0.531,0.849,0.941,0.894
swav_knn_diff,0.615,0.651,0.637,0.471,0.706,0.453
swav_finetune_20_cvae_pre_100_mc_pca,0.545,0.568,0.559,0.889,0.939,0.881
swav_finetune_20_cvae_pre_100,0.633,0.664,0.652,0.498,0.720,0.482
swav_finetune_20_cvae_pre_100_knn_diff_mc_pca,0.568,0.543,0.553,0.898,0.956,0.891
swav_finetune_20_cvae_pre_100_knn_diff,0.630,0.667,0.652,0.465,0.707,0.428


# add seacell metrics

## pancreas

In [12]:
home = '/home/icb/fatemehs.hashemig/'
SEACell_ad = sc.read_h5ad(f'{home}/pancreas_seacells.h5ad')

In [13]:
bk, lk = 'tech', 'celltype'

ds = 'pancreas'
save_pca_harmoney_metrics(SEACell_ad, bk, lk, ds, "seacell_pca")

2025-09-08 14:39:55,202 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO:harmonypy:Computing initial centroids with sklearn.KMeans...
2025-09-08 14:39:55,915 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO:harmonypy:sklearn.KMeans initialization complete.
2025-09-08 14:39:55,962 - harmonypy - INFO - Iteration 1 of 10
INFO:harmonypy:Iteration 1 of 10
2025-09-08 14:39:56,023 - harmonypy - INFO - Iteration 2 of 10
INFO:harmonypy:Iteration 2 of 10
2025-09-08 14:39:56,047 - harmonypy - INFO - Iteration 3 of 10
INFO:harmonypy:Iteration 3 of 10
2025-09-08 14:39:56,079 - harmonypy - INFO - Iteration 4 of 10
INFO:harmonypy:Iteration 4 of 10
2025-09-08 14:39:56,112 - harmonypy - INFO - Iteration 5 of 10
INFO:harmonypy:Iteration 5 of 10
2025-09-08 14:39:56,138 - harmonypy - INFO - Iteration 6 of 10
INFO:harmonypy:Iteration 6 of 10
2025-09-08 14:39:56,164 - harmonypy - INFO - Iteration 7 of 10
INFO:harmonypy:Iteration 7 of 10
2025-09-08 14:39:56,190 - 

Skipped cell type schwann, due to < 10 cells
Skipped cell type endothelial, due to < 10 cells
Skipped cell type macrophage, due to < 10 cells
Skipped cell type quiescent_stellate, due to < 10 cells
Skipped cell type mast, due to < 10 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/9 [00:00<?, ?it/s]

Skipped batch celseq2, due to < 100 cells
Skipped batch fluidigmc1, due to < 100 cells
Skipped batch smartseq2, due to < 100 cells
Skipped batch inDrop2, due to < 100 cells
Skipped batch celseq, due to < 100 cells
Skipped batch inDrop1, due to < 100 cells
Skipped batch inDrop4, due to < 100 cells
Skipped batch inDrop3, due to < 100 cells
Skipped batch smarter, due to < 100 cells


ValueError: No objects to concatenate

## immune

In [15]:
seacell_agg = sc.read_h5ad(f'{home}/models/pbmc-immune/seacell_agg.h5ad')
seacell_sc = sc.read_h5ad(f'{home}/models/pbmc-immune/seacell_sc.h5ad')

In [16]:
get_seacell_metrics(seacell_agg, seacell_sc, 'study', 'final_annotation', 'pbmc-immune')

2025-09-08 14:40:32,805 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO:harmonypy:Computing initial centroids with sklearn.KMeans...
2025-09-08 14:40:33,559 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO:harmonypy:sklearn.KMeans initialization complete.
2025-09-08 14:40:33,561 - harmonypy - INFO - Iteration 1 of 10
INFO:harmonypy:Iteration 1 of 10
2025-09-08 14:40:33,598 - harmonypy - INFO - Iteration 2 of 10
INFO:harmonypy:Iteration 2 of 10
2025-09-08 14:40:33,617 - harmonypy - INFO - Iteration 3 of 10
INFO:harmonypy:Iteration 3 of 10
2025-09-08 14:40:33,634 - harmonypy - INFO - Iteration 4 of 10
INFO:harmonypy:Iteration 4 of 10
2025-09-08 14:40:33,648 - harmonypy - INFO - Converged after 4 iterations
INFO:harmonypy:Converged after 4 iterations


Skipped cell type CD10+ B cells, due to < 10 cells
Skipped cell type CD8+ T cells, due to < 10 cells
Skipped cell type Plasma cells, due to < 10 cells
Skipped cell type Megakaryocyte progenitors, due to < 10 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/5 [00:00<?, ?it/s]

Skipped batch 10X, due to < 100 cells
Skipped batch Sun, due to < 100 cells
Skipped batch Freytag, due to < 100 cells
Skipped batch Villani, due to < 100 cells


Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]WARNING:jax._src.xla_bridge:An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.

Metrics:  60%|██████    | 6/10 [00:16<00:11,  2.81s/it, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD8+ T cells consists of a single batch or is too small. Skip.                                            
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Plasma cells consists of a single batch or is too small. Skip.                                            



Metrics:   0%|          | 0/10 [00:00<?, ?it/s]
                                                                                         
Metrics:  60%|██████    | 6/10 [00:00<00:01,  2.18it/s, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD8+ T cells consists of a single batch or is too small. Skip.                                            
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Plasma cells consists of a single batch or is too small. Skip.                                            



Embeddings: 100%|██████████| 2/2 [00:26<00:00, 13.14s/it]atch correction: pcr_comparison]    

                                                                                         

(                     Isolated labels KMeans NMI KMeans ARI Silhouette label  \
 Embedding                                                                     
 seacell_pca                 0.701739    0.77803   0.588735         0.582683   
 seacell_pca_harmoney        0.625824    0.80267   0.738461         0.607853   
 scpoli                      0.586442   0.546949   0.304108         0.525358   
 t2_cvae_0.01_prop_1         0.589887   0.657145   0.402389         0.567761   
 X_pca                       0.599962   0.587749   0.387142         0.536031   
 X_pca_harmoney              0.575475   0.688179   0.535855         0.568512   
 X_scvi                      0.623681   0.661308   0.441094         0.569204   
 
                          cLISI Silhouette batch     iLISI      KBET  \
 Embedding                                                             
 seacell_pca           0.940706         0.663277  0.144948  0.576221   
 seacell_pca_harmoney   0.95677         0.759837  0.460046  0.

# show tables

In [17]:
generate_table('~/models/pbmc-immune/')

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
seacell_pca,0.450,0.718,0.611,0.929,0.978,0.936
seacell_pca_harmoney,0.704,0.746,0.729,0.848,0.962,0.902
scpoli,0.705,0.588,0.635,0.492,0.673,0.450
t2_cvae_0.01_prop_1,0.595,0.641,0.623,0.520,0.694,0.479
X_pca,0.317,0.621,0.500,0.665,0.499,0.357
X_pca_harmoney,0.560,0.673,0.628,0.490,0.653,0.443
X_scvi,0.606,0.659,0.637,0.546,0.674,0.486


In [18]:
generate_table('~/models/pancreas/')

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
scpoli-new_ds_panc,0.602,0.703,0.662,0.509,0.735,0.460
corrected-ds_ds_panc_NP_220_cvae_0.01_prop_1_bs_128,0.537,0.682,0.624,0.666,0.805,0.643
X_pca,0.352,0.672,0.544,0.915,0.962,0.921
X_pca_harmoney,0.681,0.776,0.738,0.916,0.971,0.932
X_scvi,0.588,0.736,0.676,0.530,0.700,0.424


In [31]:
df = pd.read_csv('~/models/pbmc-immune/scgraph.csv', index_col=0)
df.style.apply(highlight_max_second, axis=0)

,Rank-PCA,Corr-PCA,Corr-Weighted
seacell_pca,0.929487,0.977643,0.936127
seacell_pca_harmoney,0.848485,0.962185,0.902033
scpoli,0.492463,0.673433,0.449950
t2_cvae_0.01_prop_1,0.520404,0.693673,0.478531
X_pca,0.664706,0.498572,0.356963
X_pca_harmoney,0.489522,0.652790,0.442862
X_scvi,0.545588,0.674347,0.486010


In [32]:
df.drop([col for col in df.index if 'seacell' in col], axis=0).style.apply(highlight_max_second, axis=0)

,Rank-PCA,Corr-PCA,Corr-Weighted
scpoli,0.492463,0.673433,0.449950
t2_cvae_0.01_prop_1,0.520404,0.693673,0.478531
X_pca,0.664706,0.498572,0.356963
X_pca_harmoney,0.489522,0.652790,0.442862
X_scvi,0.545588,0.674347,0.486010


# compute metrics for scProto metacells - Immune ds

In [45]:
scproto_params = {
    "cvae_loss_scaler": 0.01,
    "propagation_reg": 1,
    "experiment_name": "t2",
    "debug": 1
}

In [47]:
from interpretable_ssl.trainers.scproto import *


t = SCProtoTrainer(**scproto_params)

dataset is None, loading pbmc-immune
loading pbmc-immune data
done
✅ Already subsetted to HVGs (4000 genes).


In [49]:
model = t.load_model()

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 



In [51]:
protos = model.get_prototypes()

In [76]:
batch = np.zeros((protos.shape[0],1))
batch = torch.as_tensor(batch, dtype=torch.long, device='cuda')
batch.shape

torch.Size([300, 1])

In [83]:
sizefactor = np.ones((protos.shape[0],))
sizefactor = torch.as_tensor(sizefactor, dtype=torch.long, device='cuda')
sizefactor.shape

torch.Size([300])

In [84]:
metacells = model.decode(protos, batch, sizefactor)

In [106]:
sample_proto_sim = t.encode_adata(adata, model, return_mapped = True)
sample_proto_sim.shape

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

Embedding dictionary:
 	Num conditions: [5]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 



100%|██████████| 33/33 [00:07<00:00,  4.54it/s]


torch.Size([33506, 300])

In [149]:
from interpretable_ssl.scproto_metacells import *

In [150]:
proto_labels = extract_proto_labels(adata, sample_proto_sim.detach().cpu().numpy(), ['final_annotation', 'study'])

In [136]:
metacells_adata = generate_metacell_adata(metacells, proto_labels)

In [ ]:
sc.tl.pca(metacells_adata)
# rename

In [142]:
save_metrics(metacells_adata, ['scproto_metacells_pca'], 'pbmc-immune', 'study', 'final_annotation')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


2025-09-09 06:23:44,750 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO - 09/09/25 06:23:44 - 11:02:54 - Computing initial centroids with sklearn.KMeans...
2025-09-09 06:23:46,073 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO - 09/09/25 06:23:46 - 11:02:55 - sklearn.KMeans initialization complete.
2025-09-09 06:23:46,090 - harmonypy - INFO - Iteration 1 of 10
INFO - 09/09/25 06:23:46 - 11:02:55 - Iteration 1 of 10
2025-09-09 06:23:46,124 - harmonypy - INFO - Iteration 2 of 10
INFO - 09/09/25 06:23:46 - 11:02:55 - Iteration 2 of 10
2025-09-09 06:23:46,167 - harmonypy - INFO - Iteration 3 of 10
INFO - 09/09/25 06:23:46 - 11:02:56 - Iteration 3 of 10
2025-09-09 06:23:46,195 - harmonypy - INFO - Iteration 4 of 10
INFO - 09/09/25 06:23:46 - 11:02:56 - Iteration 4 of 10
2025-09-09 06:23:46,223 - harmonypy - INFO - Converged after 4 iterations
INFO - 09/09/25 06:23:46 - 11:02:56 - Converged after 4 iterations


Skipped cell type Megakaryocyte progenitors, due to < 10 cells
Skipped cell type Monocyte-derived dendritic cells, due to < 10 cells
Skipped cell type CD10+ B cells, due to < 10 cells
Skipped cell type Erythroid progenitors, due to < 10 cells
Skipped cell type HSPCs, due to < 10 cells
Skipped cell type CD16+ Monocytes, due to < 10 cells
Skipped cell type Plasmacytoid dendritic cells, due to < 10 cells
Skipped cell type Plasma cells, due to < 10 cells
Skipped cell type Monocyte progenitors, due to < 10 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/5 [00:00<?, ?it/s]

Skipped batch 10X, due to < 100 cells
Skipped batch Sun, due to < 100 cells
Skipped batch Freytag, due to < 100 cells
Skipped batch Villani, due to < 100 cells


...

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD16+ Monocytes consists of a single batch or is too small. Skip.                                         
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     HSPCs consists of a single batch or is too small. Skip.                                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Monocyte-derived dendritic cells consists of a single batch or is too small. Skip.                        
INFO     Plasma cells consists of a single batch or is too small. Skip. 


Metrics:   0%|          | 0/10 [00:00<?, ?it/s]
                                                                                         
Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/09/25 06:24:20 - 11:03:29 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:01<00:00,  7.24it/s, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD16+ Monocytes consists of a single batch or is too small. Skip.                                         
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     HSPCs consists of a single batch or is too small. Skip.                                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Monocyte-derived dendritic cells consists of a single batch or is too small. Skip.                        
INFO     Plasma cells consists of a single batch or is too small. Skip. 


Embeddings: 100%|██████████| 2/2 [00:26<00:00, 13.06s/it]atch correction: pcr_comparison]

                                                                                         

(                               Isolated labels KMeans NMI KMeans ARI  \
 Embedding                                                              
 scProto_metacells_pca                 0.590772   0.616662   0.289849   
 scProto_metacells_pca_harmoney        0.593523   0.623641   0.253203   
 seacell_pca                           0.701739    0.77803   0.588735   
 seacell_pca_harmoney                  0.625824    0.80267   0.738461   
 scpoli                                0.586442   0.546949   0.304108   
 t2_cvae_0.01_prop_1                   0.589887   0.657145   0.402389   
 X_pca                                 0.599962   0.587749   0.387142   
 X_pca_harmoney                        0.575475   0.688179   0.535855   
 X_scvi                                0.623681   0.661308   0.441094   
 
                                Silhouette label     cLISI Silhouette batch  \
 Embedding                                                                    
 scProto_metacells_pca               

In [148]:
generate_table('~/models/pbmc-immune/')

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
scProto_metacells_pca,0.543,0.595,0.574,0.832,0.961,0.845
seacell_pca,0.450,0.718,0.611,0.929,0.978,0.936
seacell_pca_harmoney,0.704,0.746,0.729,0.848,0.962,0.902
scpoli,0.705,0.588,0.635,0.492,0.673,0.450
t2_cvae_0.01_prop_1,0.595,0.641,0.623,0.520,0.694,0.479
X_pca,0.317,0.621,0.500,0.665,0.499,0.357
X_pca_harmoney,0.560,0.673,0.628,0.490,0.653,0.443
X_scvi,0.606,0.659,0.637,0.546,0.674,0.486


# re-calc ref metrics

In [250]:
# pca. harmoney, scvi, scpoli, scproto, seacell, seacell_harmoney, scproto_mc_pca

In [255]:
ref = adata[~adata.obs.study.isin(DATASETS['pbmc-immune']['test_studies'])]
ref = ref.copy()
bk, lk = 'study', 'final_annotation'

In [265]:
ds = 'pbmc-immune'
scgraph_kwargs = {
    "thres_batch": 10,
    "thres_celltype": 10,
}


In [266]:
get_scproto_metacell_metrics(t, ref, ds, bk, lk, name_postfix = 'ref_v2', **scgraph_kwargs)

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 



100%|██████████| 29/29 [00:03<00:00,  7.53it/s]


Skipped cell type Megakaryocyte progenitors, due to < 10 cells
Skipped cell type Monocyte-derived dendritic cells, due to < 10 cells
Skipped cell type CD10+ B cells, due to < 10 cells
Skipped cell type HSPCs, due to < 10 cells
Skipped cell type CD16+ Monocytes, due to < 10 cells
Skipped cell type Plasmacytoid dendritic cells, due to < 10 cells
Skipped cell type Plasma cells, due to < 10 cells
Skipped cell type Monocyte progenitors, due to < 10 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/3 [00:00<?, ?it/s]

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scgraph_ref_v2.csv


Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/11/25 07:24:21 - 2 days, 12:03:31 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:13<00:09,  2.36s/it, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD16+ Monocytes consists of a single batch or is too small. Skip.                                         
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     HSPCs consists of a single batch or is too small. Skip.                                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Monocyte-derived dendritic cells consists of a single batch or is too small. Skip.                        
INFO     Plasma cells consists of a single batch or is too small. Skip. 


Embeddings: 100%|██████████| 1/1 [00:16<00:00, 16.50s/it]atch correction: pcr_comparison]

                                                                                         

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scib_ref_v2.csv


(               Isolated labels KMeans NMI KMeans ARI Silhouette label  \
 Embedding                                                               
 scProto_mc_pca        0.597917   0.632531   0.283255         0.534265   
 
                    cLISI Silhouette batch     iLISI      KBET  \
 Embedding                                                       
 scProto_mc_pca  0.957261           0.7509  0.266704  0.776951   
 
                Graph connectivity PCR comparison Batch correction  \
 Embedding                                                           
 scProto_mc_pca           0.958872            0.0         0.550685   
 
                Bio conservation     Total  
 Embedding                                  
 scProto_mc_pca         0.601046  0.580902  ,
                 Rank-PCA  Corr-PCA  Corr-Weighted
 scProto_mc_pca     0.875  0.972581       0.924775)

In [270]:
p = f'{home}/models/pbmc-immune/ref_seacell'
ad, SEACell_ad = sc.read_h5ad(p + '_sc.h5ad'), sc.read_h5ad(p + '_agg.h5ad')

get_seacell_metrics(SEACell_ad, ad, 'study', 'final_annotation', 'pbmc-immune', 'ref_v2', **scgraph_kwargs)

2025-09-11 07:27:19,749 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO - 09/11/25 07:27:19 - 2 days, 12:06:29 - Computing initial centroids with sklearn.KMeans...
2025-09-11 07:27:20,557 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - sklearn.KMeans initialization complete.
2025-09-11 07:27:20,564 - harmonypy - INFO - Iteration 1 of 10
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - Iteration 1 of 10
2025-09-11 07:27:20,621 - harmonypy - INFO - Iteration 2 of 10
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - Iteration 2 of 10
2025-09-11 07:27:20,652 - harmonypy - INFO - Iteration 3 of 10
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - Iteration 3 of 10
2025-09-11 07:27:20,671 - harmonypy - INFO - Iteration 4 of 10
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - Iteration 4 of 10
2025-09-11 07:27:20,686 - harmonypy - INFO - Iteration 5 of 10
INFO - 09/11/25 07:27:20 - 2 days, 12:06:30 - Iteration 5 of 10
2

Skipped cell type CD10+ B cells, due to < 10 cells
Skipped cell type CD8+ T cells, due to < 10 cells
Skipped cell type Plasma cells, due to < 10 cells
Skipped cell type Megakaryocyte progenitors, due to < 10 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/3 [00:00<?, ?it/s]

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scgraph_ref_v2.csv


Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/11/25 07:27:23 - 2 days, 12:06:32 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:01<00:02,  1.72it/s, Batch correction: kbet_per_label]

...


Metrics:   0%|          | 0/10 [00:00<?, ?it/s]
                                                                                         
Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/11/25 07:27:24 - 2 days, 12:06:34 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:01<00:01,  2.04it/s, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     CD8+ T cells consists of a single batch or is too small. Skip.                                            
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Megakaryocyte progenitors consists of a single batch or is too small. Skip.                               
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    
INFO     Plasma cells consists of a single batch or is too small. Skip.                                            



Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]atch correction: pcr_comparison]    

                                                                                         

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scib_ref_v2.csv


(                     Isolated labels KMeans NMI KMeans ARI Silhouette label  \
 Embedding                                                                     
 seacell_pca                 0.690193     0.7612    0.48162          0.59916   
 seacell_pca_harmoney        0.653794   0.797615    0.62345         0.624343   
 scProto_mc_pca              0.597917   0.632531   0.283255         0.534265   
 
                          cLISI Silhouette batch     iLISI      KBET  \
 Embedding                                                             
 seacell_pca            0.95729          0.66175  0.201272  0.555556   
 seacell_pca_harmoney  0.977364         0.771486  0.617611   0.84946   
 scProto_mc_pca        0.957261           0.7509  0.266704  0.776951   
 
                      Graph connectivity PCR comparison Batch correction  \
 Embedding                                                                 
 seacell_pca                    0.935342            0.0         0.470784   
 seacell

In [ ]:
scproto_params = {
    "cvae_loss_scaler": 0.01,
    "propagation_reg": 1,
    "experiment_name": "t2",
    "debug": 1
}
t = SCProtoTrainer(**scproto_params)
add_trainer_emb(t, ref)
scpoli_params = {
    "model": "scpoli"
}
t2 = OriginalTrainer(**scpoli_params)
add_trainer_emb(t2, ref)

In [271]:
ref = add_pca_harmoney(ref, bk, 'X_pca')
ref = add_scvi_emb(ref, [], bk)

2025-09-11 07:28:46,069 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
INFO - 09/11/25 07:28:46 - 2 days, 12:07:55 - Computing initial centroids with sklearn.KMeans...
2025-09-11 07:28:50,845 - harmonypy - INFO - sklearn.KMeans initialization complete.
INFO - 09/11/25 07:28:50 - 2 days, 12:08:00 - sklearn.KMeans initialization complete.
2025-09-11 07:28:50,957 - harmonypy - INFO - Iteration 1 of 10
INFO - 09/11/25 07:28:50 - 2 days, 12:08:00 - Iteration 1 of 10
2025-09-11 07:28:55,099 - harmonypy - INFO - Iteration 2 of 10
INFO - 09/11/25 07:28:55 - 2 days, 12:08:04 - Iteration 2 of 10
2025-09-11 07:28:59,252 - harmonypy - INFO - Iteration 3 of 10
INFO - 09/11/25 07:28:59 - 2 days, 12:08:09 - Iteration 3 of 10
2025-09-11 07:29:02,455 - harmonypy - INFO - Converged after 3 iterations
INFO - 09/11/25 07:29:02 - 2 days, 12:08:12 - Converged after 3 iterations
Multiprocessing is handled by SLURM.
GPU available: True (cuda), used: True
TPU available: False, using: 0

Epoch 100/100: 100%|██████████| 100/100 [02:39<00:00,  1.59s/it, loss=647, v_num=1]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [02:39<00:00,  1.60s/it, loss=647, v_num=1]


Multiprocessing is handled by SLURM.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-6f66a4db-c267-549f-945a-69946ecc988e]
SLURM auto-requeueing enabled. Setting signal handlers.


Epoch 1/1: 100%|██████████| 1/1 [00:02<00:00,  2.13s/it, loss=643, v_num=1]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 1/1: 100%|██████████| 1/1 [00:02<00:00,  2.13s/it, loss=643, v_num=1]


In [274]:
emb_keys = [
    "X_pca",
    "t2_cvae_0.01_prop_1",
    "scpoli",
    "X_pca_harmoney",
    "X_scvi",
]


In [275]:
save_metrics(ref, emb_keys, ds, bk, lk, name_postfix='ref_v2', **scgraph_kwargs)

... storing 'conditions_combined' as categorical


Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/3 [00:00<?, ?it/s]

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scgraph_ref_v2.csv


Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/11/25 07:43:39 - 2 days, 12:22:49 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:42<00:16,  4.16s/it, Batch correction: kbet_per_label]

...


Metrics:   0%|          | 0/10 [00:00<?, ?it/s]
                                                                                         
Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/11/25 07:46:21 - 2 days, 12:25:30 - isolated labels: no more than 1 batches per label

Metrics:  60%|██████    | 6/10 [00:06<00:02,  1.58it/s, Batch correction: kbet_per_label]

INFO     CD10+ B cells consists of a single batch or is too small. Skip.                                           
INFO     Erythrocytes consists of a single batch or is too small. Skip.                                            
INFO     Erythroid progenitors consists of a single batch or is too small. Skip.                                   
INFO     Monocyte progenitors consists of a single batch or is too small. Skip.                                    



Embeddings: 100%|██████████| 5/5 [03:01<00:00, 36.29s/it]atch correction: pcr_comparison]

                                                                                         

ref_v2 /home/icb/fatemehs.hashemig/models//pbmc-immune//scib_ref_v2.csv


(                     Isolated labels KMeans NMI KMeans ARI Silhouette label  \
 Embedding                                                                     
 X_pca                        0.59233   0.652593   0.414049         0.565817   
 t2_cvae_0.01_prop_1         0.577204   0.661628   0.383094         0.572264   
 scpoli                       0.58403   0.542031    0.28758         0.526546   
 X_pca_harmoney              0.546008   0.700868   0.474959         0.568079   
 X_scvi                      0.621211   0.709064   0.533177         0.578135   
 seacell_pca                 0.690193     0.7612    0.48162          0.59916   
 seacell_pca_harmoney        0.653794   0.797615    0.62345         0.624343   
 scProto_mc_pca              0.597917   0.632531   0.283255         0.534265   
 
                          cLISI Silhouette batch     iLISI      KBET  \
 Embedding                                                             
 X_pca                 0.997075         0.723055      

In [276]:
generate_table(f'{home}/models/pbmc-immune/', 'ref_v2')

,Batch correction,Bio conservation,Total,Rank-PCA,Corr-PCA,Corr-Weighted
X_pca,0.324,0.644,0.516,0.841,0.914,0.847
t2_cvae_0.01_prop_1,0.629,0.637,0.634,0.547,0.724,0.514
scpoli,0.773,0.584,0.660,0.481,0.685,0.442
X_pca_harmoney,0.607,0.658,0.637,0.788,0.890,0.795
X_scvi,0.675,0.688,0.683,0.556,0.733,0.545
seacell_pca,0.471,0.698,0.607,0.943,0.988,0.969
seacell_pca_harmoney,0.770,0.735,0.749,0.931,0.989,0.974
scProto_mc_pca,0.551,0.601,0.581,0.875,0.973,0.925


# compare seacell - scproto on scgraph metrics in 1 batch

In [32]:
pwd

'/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl'

In [33]:
home = '/ictstr01/home/icb/fatemehs.hashemig/'
p = f'{home}/models/cd34/ref_seacell'
ad, SEACell_ad = sc.read_h5ad(p + '_sc.h5ad'), sc.read_h5ad(p + '_agg.h5ad')

In [326]:
    sc.pp.normalize_per_cell(SEACell_ad)
    sc.pp.log1p(SEACell_ad)

In [329]:
SEACell_ad = agg_obs(SEACell_ad, ad, 'celltype')

In [342]:
SEACell_ad

AnnData object with n_obs × n_vars = 95 × 12464
    obs: 'n_counts', 'celltype', 'batch'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'raw'

In [336]:
sc.pp.highly_variable_genes(
        SEACell_ad,
        n_top_genes=1000,
    )

In [337]:
sc.tl.pca(SEACell_ad, use_highly_variable=True)

In [339]:
SEACell_ad.obs['batch'] = ['b0']*95

In [350]:
SEACell_ad.obs['batch'].nunique()

1

In [347]:
SEACell_ad.obs.batch.value_counts()

b0    95
Name: batch, dtype: int64

In [348]:
SEACell_ad.write("tmp.h5ad")
scgraph = scGraph(
    adata_path="tmp.h5ad", label_key='celltype', batch_key = 'batch', thres_celltype = 3, thres_batch=10
)
scgr_res = scgraph.main(_obsm_list=['X_pca'])

Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/1 [00:00<?, ?it/s]

In [349]:
scgr_res

,Rank-PCA,Corr-PCA,Corr-Weighted
X_pca,1.0,0.999865,0.999818


# add scproto + scproto mc scgraph metrics

In [1]:
# load scproto model trained on cd34
# generate scproto metacells
# hvg + pca -> scgraph with same params

In [9]:
p = {
    'cvae_epochs': 100,
    'dataset_id': 'cd34',
    'num_prototypes': 95,
    'batch_size': 128
}
t = SCProtoTrainer(debug=1, **p)

dataset is None, loading cd34
loading cd34 data
done
ℹ️ Found 1000 HVGs out of 12464 total genes.
1 batch dataset


In [13]:
metacells_adata = get_scproto_mc_adata(t, t.dataset.adata, 'batch', 'celltype')

Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 1000 32 10
	Mean/Var Layer in/out: 32 8
Decoder Architecture:
	First Layer in, out and cond:  8 32 10
	Output Layer in/out:  32 1000 



100%|██████████| 54/54 [00:05<00:00,  9.12it/s]


In [14]:
sc.pp.highly_variable_genes(
        metacells_adata,
        n_top_genes=1000,
    )

In [22]:
sc.tl.pca(metacells_adata, use_highly_variable=True)

In [28]:
metacells_adata.write("tmp.h5ad")
scgraph = scGraph(
    adata_path="tmp.h5ad", label_key='celltype', batch_key = 'batch', thres_celltype = 3, thres_batch=10
)
scgr_res = scgraph.main(_obsm_list=['X_pca'])

Skipped cell type CLP, due to < 3 cells
Skipped cell type pDC, due to < 3 cells
Processing batches, calcualte centroids and pairwise distances


  0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
scgr_res

,Rank-PCA,Corr-PCA,Corr-Weighted
X_pca,1.0,0.999986,0.999968


In [35]:
SEACell_ad.X.max()

1178.4344482421875

In [38]:
model = t.load_model()
protos = model.get_prototypes()
batch = np.zeros((protos.shape[0], 1))
batch = torch.as_tensor(batch, dtype=torch.long, device="cuda")
sizefactor = np.ones((protos.shape[0],))
sizefactor = torch.as_tensor(sizefactor, dtype=torch.long, device="cuda")
metacells = model.decode(protos, batch, sizefactor)

Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 1000 32 10
	Mean/Var Layer in/out: 32 8
Decoder Architecture:
	First Layer in, out and cond:  8 32 10
	Output Layer in/out:  32 1000 



In [42]:
# reproduce all results

torch.Size([95, 1000])

In [41]:
add save_metrics to trainer

save_scib
save_scgraph
save_metacell

AnnData object with n_obs × n_vars = 6881 × 1000
    obs: 'leiden', 'celltype', 'n_counts', 'batch', 'conditions_combined'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'celltype_colors', 'hvg', 'log1p', 'pca'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'

In [45]:
model = t.get_model()

Embedding dictionary:
 	Num conditions: [1]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 1000 32 10
	Mean/Var Layer in/out: 32 8
Decoder Architecture:
	First Layer in, out and cond:  8 32 10
	Output Layer in/out:  32 1000 



In [46]:
model

scProtoGMVAE(
  (scpoli_cvae): scpoli(
    (embeddings): ModuleList(
      (0): Embedding(1, 10, max_norm=1.0)
    )
    (encoder): Encoder(
      (FC): Sequential(
        (L0): CondLayers(
          (expr_L): Linear(in_features=1000, out_features=32, bias=True)
          (cond_L): Linear(in_features=10, out_features=32, bias=False)
        )
        (N0): LayerNorm((32,), eps=1e-05, elementwise_affine=False)
        (A0): ReLU()
        (D0): Dropout(p=0.05, inplace=False)
      )
      (mean_encoder): Linear(in_features=32, out_features=8, bias=True)
      (log_var_encoder): Linear(in_features=32, out_features=8, bias=True)
    )
    (decoder): Decoder(
      (FirstL): Sequential(
        (L0): CondLayers(
          (expr_L): Linear(in_features=8, out_features=32, bias=False)
          (cond_L): Linear(in_features=10, out_features=32, bias=False)
        )
        (N0): LayerNorm((32,), eps=1e-05, elementwise_affine=False)
        (A0): ReLU()
        (D0): Dropout(p=0.05, inplace=F

In [138]:
t = SCProtoTrainer(proto_init = 'random', ft_epochs=5)

dataset is None, loading pbmc-immune
loading pbmc-immune data
done
✅ Already subsetted to HVGs (4000 genes).


In [139]:
t.model = t.load_model()

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

Embedding dictionary:
 	Num conditions: [5]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 



In [92]:
t.log_wandb_loss = lambda a, b: None
t.train(5)

In [141]:
import os
os.environ["JAX_ENABLE_X64"] = "0"   # force 32-bit JAX
# optional, extra belt:
os.environ["JAX_DEFAULT_DTYPE_BITS"] = "32"


In [143]:
t.dataset.adata

AnnData object with n_obs × n_vars = 33506 × 4000
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'sample_ID', 'size_factors', 'species', 'study', 'tissue', 'conditions_combined'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'pca'
    obsm: 'swav', 'X_pca'
    varm: 'PCs'
    layers: 'counts'

In [144]:
get_scib(t.dataset.adata, ['X_pca'], 'study', 'final_annotation')

Metrics:   0%|          | 0/10 [00:00<?, ?it/s, Bio conservation: isolated_labels]INFO - 09/12/25 09:28:57 - 11:47:22 - isolated labels: no more than 1 batches per label

Embeddings:   0%|          | 0/1 [00:37<?, ?it/s]5s/it, Bio conservation: nmi_ari_cluster_labels_kmeans]


TypeError: body_fun output and input must have identical types, got
('ShapedArray(float32[16,50])', 'ShapedArray(float32[])', 'DIFFERENT ShapedArray(float32[]) vs. ShapedArray(float64[], weak_type=True)', 'ShapedArray(float64[1])').

In [1]:
from scib_metrics.benchmark import Benchmarker


In [2]:
import jax
print("enable_x64:", jax.config.read("jax_enable_x64"))
print("default dtype:", jax.numpy.array(0.0).dtype)


enable_x64: False


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


default dtype: float32


In [3]:
import SEACells

In [4]:
print("default dtype:", jax.numpy.array(0.0).dtype)


default dtype: float64


In [6]:
pwd

'/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl/notebooks'

In [8]:
import scanpy as sc
h = '/ictstr01/home/icb/fatemehs.hashemig/'
ad = sc.read_h5ad(f'{h}/data/pancreas_hvg.h5ad')

In [12]:
sc.tl.pca(ad)

In [13]:
bm = Benchmarker(
    adata=ad,
    batch_key='tech',
    label_key='celltype',
    embedding_obsm_keys=['X_pca'],  # evaluate the PCA space
)
bm.benchmark()  # runs neighbors, clustering, and metrics
results = bm.get_results(min_max_scale=False)

Embeddings:   0%|          | 0/1 [00:13<?, ?it/s]6s/it, Bio conservation: nmi_ari_cluster_labels_kmeans]


TypeError: body_fun output and input must have identical types, got
('ShapedArray(float32[14,50])', 'ShapedArray(float32[])', 'DIFFERENT ShapedArray(float32[]) vs. ShapedArray(float64[], weak_type=True)', 'ShapedArray(float64[1])').